### Payments duomenų prijungmas ir analize

##### 2.6.2. Mokėjimų duomenų integracija

Šis skyrius analizuoja 10 vartotojų, kurie buvo identifikuoti kaip anomalijos
abiem metodais (Isolation Forest ir LOF), finansinį profilį pagal mokėjimų duomenis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Modelių rezultatai
lof = pd.read_excel("Outputs/lof_results.xlsx")
iforest = pd.read_excel("Outputs/iforest_results.xlsx")
payments = pd.read_csv("payments_agg.csv")
common_anomalies = pd.read_excel("Outputs/common_risks.xlsx")

common_anomalies

lof_acc  = set(lof[lof["lof_is_anomaly"] == True]["account_number"])
if_acc   = set(iforest[iforest["iforest_is_anomaly"] == 1]["account_number"])
only_if  = if_acc  - lof_acc
only_lof = lof_acc - if_acc

df_if       = payments[payments["account_number"].isin(only_if)].copy()
df_lof      = payments[payments["account_number"].isin(only_lof)].copy()

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. PAYMENT DUOMENŲ PRIDĖJIMAS
# ─────────────────────────────────────────────────────────────────────────────

# Sujungimas
both_payments = common_anomalies.merge(payments, on="account_number", how="left")

# Išsaugomas pilnas sujungtas failas
both_payments.to_excel("Outputs/both_anomalies_with_payment_data.xlsx", index=False)

# Individuali anomalijų peržiūra
both_payments

,account_number,lof_score,lof_risk,iforest_anomaly_score,iforest_risk,num_payments,total_amount,avg_amount,max_amount,min_amount,std_amount,num_discounts,last_payment_date,first_payment_date,num_different_products,num_different_plans,user_group1,user_group2
0,02fd177ff43e28b012e98c1e7fe5a002,5.698948,Vidutinė rizika,0.746592,Vidutinė rizika,17,172.413793,10.141988,12.068966,3.448276,3.131005,1,2022-07-27 16:55:14,2020-08-30 20:28:33,1,1,TOP-UP_ONLY,TOP-UP_ONLY
1,2811e96ccdaa4c81a9fa3017c430a228,2.620160,Žema rizika,0.682570,Žema rizika,8,36.613272,4.576659,4.576659,4.576659,0.000000,0,2021-11-03 19:26:16,2020-09-12 22:18:22,1,1,TOP-UP_ONLY,TOP-UP_ONLY
2,2c8cff0439be4b658554998c3cd29785,92.445980,Aukšta rizika,0.778591,Aukšta rizika,16,48.137931,3.008621,4.301724,1.715517,1.335511,0,2022-05-27 17:20:02,2021-10-18 21:24:45,2,3,BOTH,BOTH
3,461290100f21ca2b1af653cc03776f5d,11.781552,Aukšta rizika,0.773502,Aukšta rizika,25,124.439359,4.977574,11.430206,4.565217,1.509129,1,2022-07-26 21:11:38,2020-09-24 08:22:01,3,4,BOTH,BOTH
4,59a47e8de57675065f70c44deefd42c2,2.931824,Žema rizika,0.667347,Žema rizika,2,5.990000,2.995000,4.000000,1.990000,1.421285,0,2022-04-20 14:04:44,2022-03-04 14:20:12,1,1,TOP-UP_ONLY,TOP-UP_ONLY
5,7b0ac7555cbd3c781574dbf8f1dd300a,4.646698,Vidutinė rizika,0.677571,Žema rizika,2,7.986270,3.993135,5.709382,2.276888,2.427140,1,2022-03-18 03:44:09,2022-03-16 23:42:13,2,2,BOTH,BOTH
6,8b9cbe6cf4a3f8d43123178fecacc040,7.583967,Vidutinė rizika,0.773985,Aukšta rizika,4,13.695652,3.423913,4.576659,2.276888,1.324479,1,2021-09-11 02:16:10,2021-06-20 13:40:54,2,2,BOTH,BOTH
7,b286e6d78c581434fc689fb538f5a29c,3.744525,Vidutinė rizika,0.734022,Vidutinė rizika,13,182.000000,14.000000,14.000000,14.000000,0.000000,0,2021-11-19 09:00:15,2020-10-05 10:49:21,1,1,TOP-UP_ONLY,TOP-UP_ONLY
8,b6fee32818e3b530291a7176177a2c67,6.266792,Vidutinė rizika,0.751895,Vidutinė rizika,21,95.869565,4.565217,4.565217,4.565217,0.000000,0,2022-06-17 16:39:01,2020-10-17 13:06:21,1,2,PLAN_ONLY,PLAN_ONLY_SuperCheap
9,d7afdb5f0ad3dac2f5fcd366bb545fe4,3.156219,Žema rizika,0.714827,Vidutinė rizika,12,115.054828,9.587902,17.232759,4.301724,4.485596,0,2022-07-04 16:38:14,2021-11-15 03:35:39,2,2,BOTH,BOTH


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Mokėjimų charakteristikų palyginimas: anomalijos vs. populiacija
# ─────────────────────────────────────────────────────────────────────────────
skaitiniai = ["num_payments", "total_amount", "avg_amount", "max_amount",
              "std_amount", "num_discounts", "num_different_products", "num_different_plans"]

pop_mean  = payments[skaitiniai].mean()
anom_mean = both_payments[skaitiniai].mean()
nukrypimas = ((anom_mean - pop_mean) / pop_mean.replace(0, np.nan) * 100).round(1)
just_lof = df_lof[skaitiniai].mean()
just_if  = df_if[skaitiniai].mean()

lentele_11 = pd.DataFrame({
    "Populiacija (n = 9603 vidurkis)": pop_mean.round(2),
    "Bendros anomalijos (n = 10 vidurkis)":  anom_mean.round(2),
    "Nukrypimas nuo populiacijos (%)":         nukrypimas,
    'Tik iForest (n = 87 vidurkis)': just_if.round(2),
    'Tik LOF (n = 87 vidurkis)':     just_lof.round(2)
})
print("─── Mokėjimų charakteristikų palyginimas ───")
print(lentele_11.to_string())
print()



─── Mokėjimų charakteristikų palyginimas ───
                        Populiacija (n = 9603 vidurkis)  Bendros anomalijos (n = 10 vidurkis)  Nukrypimas nuo populiacijos (%)  Tik iForest (n = 87 vidurkis)  Tik LOF (n = 87 vidurkis)
num_payments                                       6.84                                 12.00                             75.4                          16.62                       4.76
total_amount                                      34.72                                 80.22                            131.1                         104.16                      21.82
avg_amount                                         5.52                                  6.13                             10.9                           8.39                       4.74
max_amount                                         7.44                                  8.25                             10.8                          16.57                       6.40
std_amount                    

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Atsiskaitymo modelių pasiskirstymas pagal aptikimo metodą
# ─────────────────────────────────────────────────────────────────────────────

tipai_rodyti = ["PLAN_ONLY", "BOTH", "TOP-UP_ONLY", "OTHER", "ZERO_PAYMENTS_ONLY"]

def grupė_eilutė(df, n):
    counts = df["user_group1"].value_counts()
    return {t: f"{counts.get(t,0)} ({counts.get(t,0)/n*100:.1f} %)" for t in tipai_rodyti}

pop_row  = {t: f"{(payments['user_group1']==t).sum()} ({(payments['user_group1']==t).sum()/len(payments)*100:.1f} %)" for t in tipai_rodyti}
if_row   = grupė_eilutė(df_if,   len(df_if))
lof_row  = grupė_eilutė(df_lof,  len(df_lof))
both_row = grupė_eilutė(both_payments, len(both_payments))

lentele_12 = pd.DataFrame({
    f"Populiacija (n={len(payments)})":  pop_row,
    f"Tik iForest (n={len(df_if)})":     if_row,
    f"Tik LOF (n={len(df_lof)})":        lof_row,
    f"Aptikti abiem (n={len(both_payments)})": both_row,
}).T
print("─── Atsiskaitymo modelių pasiskirstymas ───")
print(lentele_12.to_string())
print()



─── Atsiskaitymo modelių pasiskirstymas ───
                          PLAN_ONLY           BOTH    TOP-UP_ONLY        OTHER ZERO_PAYMENTS_ONLY
Populiacija (n=9603)  4281 (44.6 %)  2271 (23.6 %)  2631 (27.4 %)  392 (4.1 %)         28 (0.3 %)
Tik iForest (n=87)      41 (47.1 %)    36 (41.4 %)      6 (6.9 %)    4 (4.6 %)          0 (0.0 %)
Tik LOF (n=87)          37 (42.5 %)    19 (21.8 %)    30 (34.5 %)    0 (0.0 %)          1 (1.1 %)
Aptikti abiem (n=10)     1 (10.0 %)     5 (50.0 %)     4 (40.0 %)    0 (0.0 %)          0 (0.0 %)

